In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
dir = '/Users/emudr/Desktop/data/PPMI_LEDD_Project/'
outdir = '/Users/emudr/Desktop/'
out = '/Users/emudr/Desktop/df_sim.csv'
file = 'simulate_data.csv'
df = pd.read_csv(dir+file)

In [3]:
df['original_LEDDSTRMG'] = df['LEDDSTRMG']
df['original_LEDDOSE'] = df['LEDDOSE']
df['original_LEDDOSFRQ'] = df['LEDDOSFRQ']


# Example dose options, max, and frequency
drug_rules = {
    'carbidopa levodopa entacapone': {'strengths': [10, 25, 50, 100, 200], 'max_mg': 2000, 'freq_range': (3, 5)},
    'carbidopa/levodopa': {'strengths': [10, 25, 50, 100, 200, 250], 'max_mg': 10000, 'freq_range': (3, 40)},
    'controlled-release levodopa': {'strengths': [25, 50, 100, 200, 400], 'max_mg': 10000, 'freq_range': (2, 100)},
    'duodopa' : {'strengths' : [20], 'max_mg' : 4000, 'freq_range' : (1, 5000)}, 
    'levodopa': {'strengths': [50, 100, 125, 250], 'max_mg': 2400, 'freq_range': (3, 25)},
    'inbrija': {'strengths': [42], 'max_mg': 84, 'freq_range': (1, 5)},
    "entacapone": {'strengths': [200], 'max_mg': 2000, 'freq_range': (1, 4)},
    "opicapone": {'strengths': [50], 'max_mg': 50, 'freq_range': (1, 1)},
    "tolcapone": {'strengths': [100, 200], 'max_mg': 600, 'freq_range': (1, 3)},
    "pramipexole": {'strengths': [0.125, 0.25, 0.5, 0.75, 1, 1.5], 'max_mg': 900, 'freq_range': (1, 18)},
    "ropinirole": {'strengths': [0.25, 0.5, 1, 2, 4, 8], 'max_mg': 600, 'freq_range': (1, 20)},
    "rotigotine": {'strengths': [2, 4, 6, 8, 16], 'max_mg': 730, 'freq_range': (1, 5)},  # patch daily
    "piribedil": {'strengths': [50, 100], 'max_mg': 200, 'freq_range': (1, 3)},
    "selegiline": {'strengths': [5, 10], 'max_mg': 100, 'freq_range': (1, 10)},
    "rasagiline": {'strengths': [0.5, 1], 'max_mg': 300, 'freq_range': (1, 200)},
    "safinamide": {'strengths': [50, 100], 'max_mg': 10000, 'freq_range': (1, 100)},
    "istradefylline": {'strengths': [20, 40], 'max_mg': 80, 'freq_range': (1, 2)},
    "amantadine": {'strengths': [100, 200], 'max_mg': 1200, 'freq_range': (1, 12)},
    "apomporphine": {'strengths': [1, 5], 'max_mg': 1000, 'freq_range': (1, 156)},
    "gocovri": {'strengths': [137, 274], 'max_mg': 500, 'freq_range': (1, 1)},
    "osmolex": {'strengths': [129, 258, 322], 'max_mg': 622, 'freq_range': (1, 5)}
}

def backfill_ledd_priority(row, drug_rules, tolerance=0.22):
    """
    Backfill LEDD components while respecting priority:
    1. Keep LEDDSTRMG if exists
    2. Adjust LEDDOSE first
    3. Then adjust LEDDOSFRQ
    4. Adjust LEDDSTRMG only if needed
    """
    if pd.isna(row['LEDD']) or pd.isna(row['conversion_factor']):
        return row[['LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ']]
    
    drug = row['med_canonical']
    if drug not in drug_rules:
        return row[['LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ']]
    
    rules = drug_rules[drug]
    strengths = rules['strengths']
    freq_min, freq_max = rules['freq_range']
    max_mg = rules['max_mg']
    target_base = row['LEDD']
    
    # Start with existing values
    strength = row['LEDDSTRMG']
    dose = row['LEDDOSE'] if not pd.isna(row['LEDDOSE']) else 1.0
    freq = row['LEDDOSFRQ'] if not pd.isna(row['LEDDOSFRQ']) else 1.0
    
    # 1️⃣ Check if existing combination works
    total = (strength if not pd.isna(strength) else 1.0) * dose * freq *  row['conversion_factor']
        
    if np.isclose(total, target_base, rtol=tolerance) :
        return row[['LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ']]
    
    best = None
    best_error = np.inf
    
    dose_candidates = [0.25, 0.5, 1, 2, 3]
    freq_candidates = range(freq_min, freq_max + 1)
    strength_options = strengths if pd.isna(strength) else [strength]
    
    # 2️⃣ Priority: try to adjust only dose first
    for s in strength_options:
        for f in freq_candidates:
            for d in dose_candidates:
                total = s * d * f * row['conversion_factor']
                if total > max_mg:
                    continue
                error = abs(total - target_base)
                if error < best_error:
                    best_error = error
                    best = (s, d, f)
    
    # 3️⃣ If strength exists, try keeping it and adjust dose/freq only
    if not pd.isna(strength):
        for f in freq_candidates:
            for d in dose_candidates:
                total = strength * d * f * row['conversion_factor']
                if total > max_mg:
                    continue
                error = abs(total - target_base)
                if error < best_error:
                    best_error = error
                    best = (strength, d, f)
        # Then try keeping strength and dose, adjust freq only
        for f in freq_candidates:
            total = strength * dose * f * row['conversion_factor']
            if total > max_mg:
                continue
            error = abs(total - target_base)
            if error < best_error:
                best_error = error
                best = (strength, dose, f)
    
    if best is None:
        # fallback to existing values
        return row[['LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ']]
    
    return pd.Series({
        'LEDDSTRMG': best[0],
        'LEDDOSE': best[1],
        'LEDDOSFRQ': best[2]
    })

    
cols = ['LEDDSTRMG', 'LEDDOSE', 'LEDDOSFRQ']

df[cols] = df.apply(
    lambda r: backfill_ledd_priority(r, drug_rules),
    axis=1
)

df['LEDD_simulated'] = (
    df['LEDDSTRMG'] *
    df['LEDDOSE'].fillna(1.0) *
    df['LEDDOSFRQ'].fillna(1.0) *
    df['conversion_factor']
)


df['LEDD_error'] = df['LEDD_simulated'] - df['LEDD']

temp = df[['LEDTRT', 'med_canonical', 'original_LEDDSTRMG', 'LEDDSTRMG', 'original_LEDDOSE', 'LEDDOSE', 'original_LEDDOSFRQ', 'LEDDOSFRQ', 'LEDD_calculated', 'LEDD_simulated','LEDD', 'LEDD_error', 'conversion_factor']]
temp.to_csv(outdir + 'temp.csv')
print(df['LEDD_error'].describe())

janetl


count    5263.000000
mean        0.370432
std         4.110764
min       -50.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        80.000000
Name: LEDD_error, dtype: float64


NameError: name 'janetl' is not defined